# Solutions · 11 · Ion exchange and chromatography

Worked solutions to the exercises of [notebook 11](../notebooks/11_ion_exchange_and_chromatography.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engsep is missing): install it from GitHub
    import engsep
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engsep"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engsep import adsorption as ad
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

A deioniser uses a cation resin (1.8 eq/L) and an anion resin (1.2 eq/L) in series to treat water with 3 meq/L of cations and 3 meq/L of anions at 5 m3/h. Size each bed for a 24 h cycle at 75 % utilisation, and compute the acid and base needed for regeneration at 200 % of stoichiometry.

In [2]:
flow, hours, util = 5.0, 24.0, 0.75
for name, cap, load, regen, regen_eq_L in (("cation", 1.8, 3e-3, "HCl (2 M)", 2.0), ("anion", 1.2, 3e-3, "NaOH (2 M)", 2.0)):
    eq_per_cycle = load * 1e3 * flow * hours                            # equivalents fed per cycle
    V = eq_per_cycle / (cap * 1e3 * util)                               # m3 of resin
    regen_vol = 2.0 * V * cap * 1e3 / (regen_eq_L * 1e3)               # m3 of regenerant at 200 % of stoichiometry
    print(f"{name} bed: {eq_per_cycle:.0f} eq per cycle -> {V:.2f} m3 of resin; regeneration with {regen_vol:.2f} m3 of {regen}")

cation bed: 360 eq per cycle -> 0.27 m3 of resin; regeneration with 0.48 m3 of HCl (2 M)
anion bed: 360 eq per cycle -> 0.40 m3 of resin; regeneration with 0.48 m3 of NaOH (2 M)


A deioniser is two softeners in series with acid and base as regenerants; the beds are sized on the
equivalents fed per cycle over the usable capacity, and the regenerant demand is twice the stoichiometric
amount. Mixed-bed polishers and reverse osmosis pretreatment exist to shrink these numbers.

## Exercise 2

Two proteins have retention factors 1.5 and 1.8 on a 20 cm column with a plate height of 20 um. Find the resolution. How long must the column be for R_s = 1.5, and how does the run time scale?

In [3]:
t0, H, L = 1.0, 20e-6, 0.20
for kA, kB in ((1.5, 1.8),):
    N = L / H
    tA, tB = t0 * (1 + kA), t0 * (1 + kB)
    Rs = (tB - tA) / (2 * (tA + tB) / np.sqrt(N))
    print(f"{L*100:.0f} cm column, N = {N:.0f}: R_s = {Rs:.2f}")
    L_needed = L * (1.5 / Rs) ** 2
    print(f"R_s = 1.5 needs only {L_needed*100:.0f} cm (resolution scales with sqrt(L)): the 20 cm column over-resolves, and a run {L_needed/L:.1f} x as long would do")

20 cm column, N = 10000: R_s = 2.83
R_s = 1.5 needs only 6 cm (resolution scales with sqrt(L)): the 20 cm column over-resolves, and a run 0.3 x as long would do


Resolution grows only as the square root of column length while the run time grows linearly: the 20 cm
column resolves these proteins nearly twice as well as needed, and a column a third as long would do the
job in a third of the time. Better selectivity (a larger difference in retention factors) is worth far more
than a longer column; when the selectivity is there, shorten the column.

## Exercise 3

**Implement it yourself:** simulate the two chromatographic peaks as Gaussians for N = 500 and compute the resolution from the peak widths at half height (w = 2.355 sigma) instead of the formula.

In [4]:
def peak(t, t_R, N):
    sigma = t_R / np.sqrt(N)
    return np.exp(-0.5 * ((t - t_R) / sigma) ** 2)
t = np.linspace(100, 300, 20001)
N, t0 = 500, 60.0
tA, tB = t0 * 3.0, t0 * 3.4
def width_half(t_R):
    p = peak(t, t_R, N)
    above = t[p >= 0.5]
    return above[-1] - above[0]
wA, wB = width_half(tA), width_half(tB)
Rs_w = 1.18 * (tB - tA) / (wA + wB)
print(f"half-height widths {wA:.2f} and {wB:.2f} s -> R_s = {Rs_w:.3f}; formula: {(tB - tA) / (2 * (tA + tB) / np.sqrt(N)):.3f}")

half-height widths 18.94 and 21.48 s -> R_s = 0.701; formula: 0.699


Measuring the widths at half height and using $R_s = 1.18\,\Delta t/(w_A + w_B)$ gives the same resolution as
the plate-number formula - which is how resolution is read from a real chromatogram.